# Agente de voz local con Transformers

Pipeline experimental del TFM:

```text
Microfono -> faster-whisper (STT) -> Transformer (LLM) -> TTS -> Altavoces
```

El LLM se ejecuta directamente con Hugging Face Transformers. Piper es el backend TTS ejecutable por defecto porque el proyecto ya incluye una voz española local. F5-TTS y Fish Speech quedan como alternativas para una fase posterior con versiones fijadas.

## 1. Preparacion

Selecciona el kernel `Python (VoiceAgents)`. Si faltan dependencias, ejecuta en una terminal con ese entorno activo:

```powershell
python -m pip install transformers accelerate sentencepiece
```

La primera ejecucion descargara el modelo desde Hugging Face. Se usa Qwen2.5 1.5B para reducir el tiempo de respuesta en CPU. Para comparar calidad se podra cambiar posteriormente a una variante mayor.

In [ ]:
import importlib.util
import platform
import sys
import time
from pathlib import Path

print(f"Python: {sys.version.split()[0]}")
print(f"Ejecutable: {sys.executable}")
print(f"Sistema: {platform.platform()}")
for module_name, package_name in {
    "torch": "PyTorch",
    "transformers": "Transformers",
    "accelerate": "Accelerate",
    "faster_whisper": "faster-whisper",
    "sounddevice": "sounddevice",
    "soundfile": "soundfile",
}.items():
    status = "disponible" if importlib.util.find_spec(module_name) else "FALTA"
    print(f"{package_name}: {status}")

PIPER_MODEL = Path("../models/piper/es_ES-davefx-medium.onnx")
print(f"Modelo Piper: {'disponible' if PIPER_MODEL.exists() else 'no encontrado'}")

Python: 3.12.10
Ejecutable: c:\Users\oitav\Documents\VIU\TFM\voice-agents\voiceagent\Scripts\python.exe
Sistema: Windows-11-10.0.26200-SP0
PyTorch: disponible
Transformers: disponible
Accelerate: disponible
faster-whisper: disponible
sounddevice: disponible
soundfile: disponible
Modelo Piper: disponible


In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_NEW_TOKENS = 32
TEMPERATURE = 0.0
SYSTEM_PROMPT = "Eres un asistente de voz. Responde en español con una sola respuesta breve de una o dos frases. No muestres razonamientos ni análisis internos."
TTS_BACKEND = "piper"
AUDIO_DIR = Path("../models/audio")
AUDIO_DIR.mkdir(parents=True, exist_ok=True)
print(f"Modelo: {MODEL_NAME}; TTS: {TTS_BACKEND}; max tokens: {MAX_NEW_TOKENS}")

Modelo: Qwen/Qwen2.5-1.5B-Instruct; TTS: piper; max tokens: 32


## 2. LLM Transformer

Esta es la primera prueba funcional. Si falla, revisa la instalacion, el acceso a Hugging Face o la memoria antes de continuar con audio.

In [35]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

load_started = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if torch.cuda.is_available():
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        dtype=torch.float16,
        device_map="cuda",
    )
    MODEL_DEVICE = torch.device("cuda")
    print(f"GPU CUDA detectada: {torch.cuda.get_device_name(0)}")
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        dtype=torch.float32,
    )
    MODEL_DEVICE = torch.device("cpu")
    print("CUDA no disponible: modelo cargado en RAM, sin offload a disco.")

model.eval()
print(f"Dispositivo: {MODEL_DEVICE}")
print(f"Carga: {time.perf_counter() - load_started:.2f} s")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

CUDA no disponible: modelo cargado en RAM, sin offload a disco.
Dispositivo: cpu
Carga: 10.99 s


In [23]:
def generate_response(user_text: str, history: list[dict[str, str]] | None = None) -> tuple[str, float]:
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    messages.extend(history or [])
    messages.append({"role": "user", "content": user_text})
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(MODEL_DEVICE)
    started = time.perf_counter()
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = outputs[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip(), time.perf_counter() - started

response, llm_seconds = generate_response("Explica en una frase que es un agente de voz local.")
print(response)
print(f"Latencia LLM: {llm_seconds:.2f} s")

Es un asistente digital que proporciona información y ayuda a través del habla, generalmente dentro de su área geográfica específica.
Latencia LLM: 20.78 s


In [24]:
from faster_whisper import WhisperModel

stt_model = WhisperModel("base", device="cpu", compute_type="int8")

def transcribe_audio(audio_path: str) -> tuple[str, float]:
    started = time.perf_counter()
    segments, _ = stt_model.transcribe(audio_path, language="es", vad_filter=True)
    text = " ".join(segment.text.strip() for segment in segments).strip()
    return text, time.perf_counter() - started

print("STT preparado con faster-whisper/base.")

STT preparado con faster-whisper/base.


## 3. TTS y orquestador

Piper se ejecuta con el modelo local incluido. Para utilizar F5-TTS o Fish Speech, se debe crear un adaptador concreto para la version instalada; sus APIs y formatos de salida no son intercambiables automaticamente.

In [25]:
import subprocess
from IPython.display import Audio, display

def synthesize_piper(text: str, output_path: str) -> tuple[str, float]:
    if not PIPER_MODEL.exists():
        raise FileNotFoundError(f"No existe el modelo Piper: {PIPER_MODEL}")
    output_file = Path(output_path).resolve()
    output_file.parent.mkdir(parents=True, exist_ok=True)
    started = time.perf_counter()
    result = subprocess.run([sys.executable, "-m", "piper", "--model", str(PIPER_MODEL), "--output_file", str(output_file)], input=text + "\n", text=True, encoding="utf-8", capture_output=True)
    if result.returncode != 0:
        raise RuntimeError(result.stderr.strip() or result.stdout.strip())
    return str(output_file), time.perf_counter() - started

def synthesize_audio(text: str, output_path: str) -> tuple[str, float]:
    if TTS_BACKEND == "piper":
        return synthesize_piper(text, output_path)
    raise ValueError("Backend TTS no implementado: añade un adaptador versionado para F5-TTS o Fish Speech.")

class VoiceAgent:
    def __init__(self) -> None:
        self.history: list[dict[str, str]] = []

    def process_audio(self, audio_path: str, output_path: str) -> dict[str, object]:
        text, stt_seconds = transcribe_audio(audio_path)
        if not text:
            return {"text": "", "response": "", "latency": {"stt": stt_seconds}}
        answer, llm_seconds = generate_response(text, self.history)
        self.history.extend([{"role": "user", "content": text}, {"role": "assistant", "content": answer}])
        audio_file, tts_seconds = synthesize_audio(answer, output_path)
        return {"text": text, "response": answer, "audio_path": audio_file, "latency": {"stt": stt_seconds, "llm": llm_seconds, "tts": tts_seconds, "total": stt_seconds + llm_seconds + tts_seconds}}

agent = VoiceAgent()
print("Orquestador preparado.")

Orquestador preparado.


In [ ]:
INPUT_AUDIO = Path("../models/audio/prueba_kernel.wav")
if not INPUT_AUDIO.exists():
    raise FileNotFoundError(f"No se encuentra el audio: {INPUT_AUDIO}")
result = agent.process_audio(str(INPUT_AUDIO), str(AUDIO_DIR / "respuesta_transformers.wav"))
print(f"Texto: {result['text']}")
print(f"Respuesta: {result['response']}")
print(f"Latencias: {result['latency']}")
display(Audio(result["audio_path"]))

Texto: Hola, esta es una prueba de voz.
Respuesta: ¡Hola! Estoy listo para ayudarte.
Latencias: {'stt': 1.6275275999942096, 'llm': 11.606000500003574, 'tts': 3.3753768999886233, 'total': 16.608904999986407}


## 4. Evaluacion y ampliaciones

Registra el modelo exacto, versiones, dispositivo, memoria, transcripcion, respuesta y latencias. Repite las mismas frases para comparar Ollama y Transformers.

La siguiente fase puede añadir grabacion desde microfono, VAD, streaming, interrupciones y adaptadores versionados para F5-TTS y Fish Speech.

## 5. Conversacion mediante el microfono

La grabacion no tiene una duracion fija. Cada turno funciona asi:

```text
Pulsar Hablar -> hablar libremente -> pulsar Parar -> faster-whisper -> Transformer -> Piper -> respuesta hablada
```

Tambien se detiene automaticamente despues de detectar silencio, siempre que primero haya detectado voz. El boton queda disponible para iniciar el siguiente turno cuando termina el procesamiento.

In [ ]:
import ipywidgets as widgets
import numpy as np
import sounddevice as sd
import soundfile as sf
import threading
import time
from IPython.display import Audio, clear_output, display

MICROPHONE_AUDIO = AUDIO_DIR / "microfono_transformers.wav"
MICROPHONE_RESPONSE = AUDIO_DIR / "respuesta_microfono_transformers.wav"
SAMPLE_RATE = 16_000
BLOCK_SIZE = 1024
SILENCE_SECONDS = 1.2
MAX_RECORD_SECONDS = 60
SILENCE_THRESHOLD = 0.015
recording_active = threading.Event()
stop_recording = threading.Event()
turn_lock = threading.Lock()


def record_until_stop_or_silence(output_path: Path) -> Path:
    print("Grabando... habla libremente y pulsa Parar cuando termines.")
    audio_blocks: list[np.ndarray] = []
    speech_detected = False
    silence_started: float | None = None
    started = time.perf_counter()
    recording_active.set()

    try:
        with sd.InputStream(
            samplerate=SAMPLE_RATE,
            blocksize=BLOCK_SIZE,
            channels=1,
            dtype="float32",
        ) as stream:
            while not stop_recording.is_set():
                block, overflowed = stream.read(BLOCK_SIZE)
                block = np.asarray(block, dtype="float32")
                audio_blocks.append(block.copy())
                level = float(np.sqrt(np.mean(np.square(block))))

                if level >= SILENCE_THRESHOLD:
                    speech_detected = True
                    silence_started = None
                elif speech_detected:
                    silence_started = silence_started or time.perf_counter()
                    if time.perf_counter() - silence_started >= SILENCE_SECONDS:
                        print("Silencio detectado; procesando...")
                        break

                if time.perf_counter() - started >= MAX_RECORD_SECONDS:
                    print("Se alcanzo el limite de seguridad; procesando...")
                    break
    except Exception as error:
        raise RuntimeError(
            "No se pudo acceder al microfono. Comprueba el dispositivo de entrada y los permisos de Windows."
        ) from error
    finally:
        recording_active.clear()
        stop_recording.clear()

    if not audio_blocks:
        raise RuntimeError("No se recibieron datos del microfono.")
    recording = np.concatenate(audio_blocks, axis=0)
    sf.write(output_path, recording, SAMPLE_RATE)
    peak = float(np.max(np.abs(recording)))
    print(f"Audio guardado | duracion: {len(recording) / SAMPLE_RATE:.1f} s | nivel maximo: {peak:.4f}")
    if not speech_detected:
        print("No se detecto voz. Puedes pulsar Hablar para intentarlo de nuevo.")
    return output_path


def play_audio(audio_path: str) -> None:
    audio_data, sample_rate = sf.read(audio_path, dtype="float32")
    sd.play(audio_data, sample_rate)
    sd.wait()


microphone_button = widgets.Button(
    description="Hablar",
    icon="microphone",
    tooltip="Pulsa para iniciar; pulsa de nuevo para parar y procesar",
    button_style="success",
)
conversation_output = widgets.Output()
turn_lock = threading.Lock()


def process_microphone_turn() -> None:
    try:
        with conversation_output:
            clear_output(wait=True)
            microphone_audio = record_until_stop_or_silence(MICROPHONE_AUDIO)
            microphone_button.description = "Procesando..."
            microphone_result = agent.process_audio(
                str(microphone_audio),
                str(MICROPHONE_RESPONSE),
            )
            text = microphone_result["text"]
            print(f"Texto reconocido: {text}")
            if not text:
                print("No se ha detectado voz. Pulsa Hablar e intenta de nuevo.")
            else:
                print(f"Respuesta: {microphone_result['response']}")
                print(f"Latencias: {microphone_result['latency']}")
                display(Audio(microphone_result["audio_path"]))
                print("Reproduciendo por los altavoces...")
                play_audio(microphone_result["audio_path"])
    except Exception as error:
        with conversation_output:
            print(f"Error durante el turno: {error}")
    finally:
        microphone_button.description = "Hablar"
        microphone_button.disabled = False
        microphone_button.button_style = "success"
        turn_lock.release()


def on_microphone_button_clicked(button: widgets.Button) -> None:
    if recording_active.is_set():
        stop_recording.set()
        microphone_button.description = "Procesando..."
        microphone_button.button_style = "warning"
        return

    if not turn_lock.acquire(blocking=False):
        return
    microphone_button.description = "Parar"
    microphone_button.icon = "stop"
    microphone_button.button_style = "danger"
    threading.Thread(target=process_microphone_turn, daemon=True).start()


microphone_button.on_click(on_microphone_button_clicked)
display(widgets.VBox([microphone_button, conversation_output]))
print("Pulsa Hablar para iniciar y Parar para enviar el turno.")

Pulsa Hablar para iniciar y Parar para enviar el turno.
